In [ ]:

import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import kagglehub

path = kagglehub.dataset_download("nyagami/ea-sports-fc-25-database-ratings-and-stats")
csv_path = os.path.join(path, "all_players.csv")

df = pd.read_csv(csv_path)
print(f"Загружено строк: {len(df)}")

#Cleaning
cols_to_drop = ["Unnamed: 0.1", "Unnamed: 0", "Rank", "Name", "url", "Potential", "potential"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# "182cm / 6'0\"" -> 182.0   |   "75kg / 165lb" -> 75.0
df["Height"] = df["Height"].astype(str).str.extract(r"([\d.]+)\s*cm").astype(float)
df["Weight"] = df["Weight"].astype(str).str.extract(r"([\d.]+)\s*kg").astype(float)

df["Position"] = df["Position"].fillna("CM").str.upper().str.strip()

cat_cols = ["Preferred foot", "Alternative positions", "Nation", "League", "Team", "play style"]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")

gk_attrs = ["GK Diving", "GK Handling", "GK Kicking", "GK Positioning", "GK Reflexes"]
field_attrs = [
    "PAC", "SHO", "PAS", "DRI", "DEF", "PHY",
    "Acceleration", "Sprint Speed", "Positioning", "Finishing", "Shot Power",
    "Long Shots", "Volleys", "Penalties", "Vision", "Crossing",
    "Free Kick Accuracy", "Short Passing", "Long Passing", "Curve", "Dribbling",
    "Agility", "Balance", "Reactions", "Ball Control", "Composure",
    "Interceptions", "Heading Accuracy", "Def Awareness", "Standing Tackle",
    "Sliding Tackle", "Jumping", "Stamina", "Strength", "Aggression",
]

for f in gk_attrs:
    df[f] = pd.to_numeric(df[f], errors="coerce").fillna(0.0)   
for f in field_attrs:
    df[f] = pd.to_numeric(df[f], errors="coerce")
    df[f] = df[f].fillna(df[f].median())

for col in ["Height", "Weight", "Age", "Weak foot", "Skill moves"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

df = pd.get_dummies(df, columns=["Preferred foot"], drop_first=True)

for col in ["Nation", "League", "Team", "Alternative positions", "play style"]:
    if col in df.columns:
        df[col] = pd.factorize(df[col])[0]

#Segmentation
def assign_segment(pos: str) -> str:
    if pos in ["CB", "LB", "RB", "CDM", "LWB", "RWB"]:
        return "Defense"
    if pos in ["CM", "CAM", "LM", "RM"]:
        return "Midfield"
    if pos in ["ST", "CF"]:
        return "Forward"
    if pos in ["LW", "RW"]:
        return "Winger"
    if pos == "GK":
        return "Goalkeeper"
    return "Midfield"

df["Segment"] = df["Position"].apply(assign_segment)
print(df["Segment"].value_counts(), "\n")

preferred_foot_dummy_cols = [c for c in df.columns if c.startswith("Preferred foot_")]

#Primary and secondary features
common_secondary_extra = ["Age", "Weak foot", "Skill moves", "Weight"] + preferred_foot_dummy_cols + \
                          ["Nation", "League", "Team", "Alternative positions", "play style"]

dribbling_group = ["Dribbling", "Agility", "Balance", "Reactions", "Ball Control", "Composure"]

forward_primary = ["PAC", "Acceleration", "Sprint Speed", "SHO", "Finishing",
                    "Shot Power", "Long Shots", "Volleys", "Penalties", "Positioning"]

SEGMENT_CONFIG = {
    "Defense": {
        "primary": ["DEF", "Interceptions", "Def Awareness", "Standing Tackle",
                    "Sliding Tackle", "Jumping", "Stamina", "Aggression", "Height"],
        "shrink": 0.35,
    },
    "Midfield": {
        "primary": ["PAS", "Vision", "Short Passing", "Long Passing", "Curve"] + dribbling_group,
        "shrink": 0.55,   
    },
    "Forward": {
        "primary": forward_primary,
        "shrink": 0.35,
    },
    "Winger": {
        "primary": forward_primary + dribbling_group,
        "shrink": 0.35,
    },
    "Goalkeeper": {
        "primary": gk_attrs + ["Height"],
        "shrink": None,  
    },
}

all_numeric_pool = list(dict.fromkeys(field_attrs + ["Height"] + common_secondary_extra))


def get_secondary_features(segment: str, primary: list[str]) -> list[str]:
    """Всё, что не профильное для сегмента (и не GK-статы, если сегмент не GK)."""
    pool = all_numeric_pool if segment != "Goalkeeper" else []
    return [f for f in pool if f not in primary]


def weak_link_feature(frame: pd.DataFrame, primary: list[str]) -> pd.Series:
    vals = frame[primary].to_numpy(dtype=float)
    return pd.Series(vals.min(axis=1) - vals.mean(axis=1), index=frame.index)


#Training
RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for seg, cfg in SEGMENT_CONFIG.items():
    seg_df = df[df["Segment"] == seg].copy()
    primary_feats = cfg["primary"]
    secondary_feats = get_secondary_features(seg, primary_feats)

    seg_df["weak_link_primary"] = weak_link_feature(seg_df, primary_feats)
    primary_feats_full = primary_feats + ["weak_link_primary"]

    y = seg_df["OVR"].astype(float)
    X_primary = seg_df[primary_feats_full]
    X_secondary = seg_df[secondary_feats] if secondary_feats else None

    idx_train, idx_test = train_test_split(seg_df.index, test_size=0.2, random_state=RANDOM_STATE)

    scaler_p = StandardScaler()
    Xp_train = scaler_p.fit_transform(X_primary.loc[idx_train])
    Xp_test = scaler_p.transform(X_primary.loc[idx_test])
    y_train, y_test = y.loc[idx_train], y.loc[idx_test]

    model_primary = GradientBoostingRegressor(
        n_estimators=150, max_depth=4, learning_rate=0.08, random_state=RANDOM_STATE
    )
    model_primary.fit(Xp_train, y_train)

    train_pred_primary = model_primary.predict(Xp_train)
    test_pred_primary = model_primary.predict(Xp_test)

    model_secondary = None
    scaler_s = None
    shrink = cfg["shrink"]

    if secondary_feats and shrink:
        oof_pred_primary = cross_val_predict(
            GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.08,
                                       random_state=RANDOM_STATE),
            Xp_train, y_train, cv=kf,
        )
        residual_train = y_train.to_numpy() - oof_pred_primary

        scaler_s = StandardScaler()
        Xs_train = scaler_s.fit_transform(X_secondary.loc[idx_train])
        Xs_test = scaler_s.transform(X_secondary.loc[idx_test])

        model_secondary = GradientBoostingRegressor(
            n_estimators=120, max_depth=3, learning_rate=0.06, random_state=RANDOM_STATE
        )
        model_secondary.fit(Xs_train, residual_train)

        train_pred_final = train_pred_primary + shrink * model_secondary.predict(Xs_train)
        test_pred_final = test_pred_primary + shrink * model_secondary.predict(Xs_test)
    else:
        train_pred_final = train_pred_primary
        test_pred_final = test_pred_primary

    train_r2 = r2_score(y_train, train_pred_final)
    test_r2 = r2_score(y_test, test_pred_final)
    test_r2_primary_only = r2_score(y_test, test_pred_primary)

    print(f"--- Сегмент: {seg} ---")
    print(f"  primary features   : {primary_feats}")
    print(f"  secondary count    : {len(secondary_feats)}  | shrink = {shrink}")
    print(f"  Test R2 (только primary)      : {test_r2_primary_only:.4f}")
    print(f"  Test R2 (primary + secondary) : {test_r2:.4f}  <- вклад непрофильных признаков виден в разнице\n")

    results[seg] = {"train_r2": train_r2, "test_r2": test_r2}

    joblib.dump(model_primary, f"model_primary_{seg}.pkl")
    joblib.dump(scaler_p, f"scaler_primary_{seg}.pkl")
    joblib.dump(primary_feats_full, f"features_primary_{seg}.pkl")
    if model_secondary is not None:
        joblib.dump(model_secondary, f"model_secondary_{seg}.pkl")
        joblib.dump(scaler_s, f"scaler_secondary_{seg}.pkl")
        joblib.dump(secondary_feats, f"features_secondary_{seg}.pkl")
    joblib.dump(shrink, f"shrink_{seg}.pkl")

print("Итог по всем сегментам:")
for seg, r in results.items():
    print(f"  {seg:10s} -> test R2 = {r['test_r2']:.4f}")



def predict_ovr(segment: str, player_row: pd.DataFrame) -> float:
    """player_row — один DataFrame-ряд с уже очищенными/закодированными признаками."""
    cfg = SEGMENT_CONFIG[segment]
    primary_feats = cfg["primary"]

    row = player_row.copy()
    row["weak_link_primary"] = weak_link_feature(row, primary_feats)
    primary_feats_full = primary_feats + ["weak_link_primary"]

    m_p = joblib.load(f"model_primary_{segment}.pkl")
    s_p = joblib.load(f"scaler_primary_{segment}.pkl")
    pred = m_p.predict(s_p.transform(row[primary_feats_full]))[0]

    shrink = joblib.load(f"shrink_{segment}.pkl")
    if shrink:
        secondary_feats = joblib.load(f"features_secondary_{segment}.pkl")
        m_s = joblib.load(f"model_secondary_{segment}.pkl")
        s_s = joblib.load(f"scaler_secondary_{segment}.pkl")
        pred += shrink * m_s.predict(s_s.transform(row[secondary_feats]))[0]

    return float(np.clip(pred, 1, 99))

Загружено строк: 17737
Segment
Defense       7369
Midfield      5159
Forward       2425
Goalkeeper    1999
Winger         785
Name: count, dtype: int64 

--- Сегмент: Defense ---
  primary features   : ['DEF', 'Interceptions', 'Def Awareness', 'Standing Tackle', 'Sliding Tackle', 'Jumping', 'Stamina', 'Aggression', 'Height']
  secondary count    : 37  | shrink = 0.35
  Test R2 (только primary)      : 0.9383
  Test R2 (primary + secondary) : 0.9522  <- вклад непрофильных признаков виден в разнице

--- Сегмент: Midfield ---
  primary features   : ['PAS', 'Vision', 'Short Passing', 'Long Passing', 'Curve', 'Dribbling', 'Agility', 'Balance', 'Reactions', 'Ball Control', 'Composure']
  secondary count    : 35  | shrink = 0.55
  Test R2 (только primary)      : 0.9603
  Test R2 (primary + secondary) : 0.9738  <- вклад непрофильных признаков виден в разнице

--- Сегмент: Forward ---
  primary features   : ['PAC', 'Acceleration', 'Sprint Speed', 'SHO', 'Finishing', 'Shot Power', 'Long Shots', '

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import kagglehub

path = kagglehub.dataset_download("nyagami/ea-sports-fc-25-database-ratings-and-stats")
csv_path = os.path.join(path, "all_players.csv")

df = pd.read_csv(csv_path)
df.head(5)


C:\Users\Асылхан\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unnamed: 0.1,Unnamed: 0,Rank,Name,OVR,PAC,SHO,PAS,DRI,DEF,...,Nation,League,Team,play style,url,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes
0,0,0,1,Kylian Mbappé,91,97,90,80,92,36,...,France,LALIGA EA SPORTS,Real Madrid,"Quick Step+, Acrobatic, Finesse Shot, Flair, R...",https://www.ea.com/games/ea-sports-fc/ratings/...,NaN,NaN,NaN,NaN,NaN
1,1,1,2,Rodri,91,66,80,86,84,87,...,Spain,Premier League,Manchester City,"Tiki Taka+, Aerial, Bruiser, Long Ball Pass, P...",https://www.ea.com/games/ea-sports-fc/ratings/...,NaN,NaN,NaN,NaN,NaN
2,2,2,4,Erling Haaland,91,88,92,70,81,45,...,Norway,Premier League,Manchester City,"Acrobatic+, Bruiser, Power Header, Power Shot,...",https://www.ea.com/games/ea-sports-fc/ratings/...,NaN,NaN,NaN,NaN,NaN
3,3,3,5,Jude Bellingham,90,80,87,83,88,78,...,England,LALIGA EA SPORTS,Real Madrid,"Relentless+, Flair, Intercept, Slide Tackle, T...",https://www.ea.com/games/ea-sports-fc/ratings/...,NaN,NaN,NaN,NaN,NaN
4,4,4,7,Vini Jr.,90,95,84,81,91,29,...,Brazil,LALIGA EA SPORTS,Real Madrid,"Quick Step+, Chip Shot, Finesse Shot, First To...",https://www.ea.com/games/ea-sports-fc/ratings/...,NaN,NaN,NaN,NaN,NaN


In [2]:
#Cleaning
cols_to_drop = ["Unnamed: 0.1", "Unnamed: 0", "Rank", "Name", "url", "Potential", "potential"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# "182cm / 6'0\"" -> 182.0   |   "75kg / 165lb" -> 75.0
df["Height"] = df["Height"].astype(str).str.extract(r"([\d.]+)\s*cm").astype(float)
df["Weight"] = df["Weight"].astype(str).str.extract(r"([\d.]+)\s*kg").astype(float)

df["Position"] = df["Position"].fillna("CM").str.upper().str.strip()

cat_cols = ["Preferred foot", "Alternative positions", "Nation", "League", "Team", "play style"]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")

gk_attrs = ["GK Diving", "GK Handling", "GK Kicking", "GK Positioning", "GK Reflexes"]
field_attrs = [
    "PAC", "SHO", "PAS", "DRI", "DEF", "PHY",
    "Acceleration", "Sprint Speed", "Positioning", "Finishing", "Shot Power",
    "Long Shots", "Volleys", "Penalties", "Vision", "Crossing",
    "Free Kick Accuracy", "Short Passing", "Long Passing", "Curve", "Dribbling",
    "Agility", "Balance", "Reactions", "Ball Control", "Composure",
    "Interceptions", "Heading Accuracy", "Def Awareness", "Standing Tackle",
    "Sliding Tackle", "Jumping", "Stamina", "Strength", "Aggression",
]

for f in gk_attrs:
    df[f] = pd.to_numeric(df[f], errors="coerce").fillna(0.0)   
for f in field_attrs:
    df[f] = pd.to_numeric(df[f], errors="coerce")
    df[f] = df[f].fillna(df[f].median())

for col in ["Height", "Weight", "Age", "Weak foot", "Skill moves"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

df = pd.get_dummies(df, columns=["Preferred foot"], drop_first=True)

for col in ["Nation", "League", "Team", "Alternative positions", "play style"]:
    if col in df.columns:
        df[col] = pd.factorize(df[col])[0]

In [3]:
#Segmentation
def assign_segment(pos: str) -> str:
    if pos in ["CB", "LB", "RB", "CDM", "LWB", "RWB"]:
        return "Defense"
    if pos in ["CM", "CAM", "LM", "RM"]:
        return "Midfield"
    if pos in ["LW", "RW", "ST", "CF"]:
        return "Attack"
    if pos == "GK":
        return "Goalkeeper"
    return "Midfield"

df["Segment"] = df["Position"].apply(assign_segment)
print(df["Segment"].value_counts(), "\n")

preferred_foot_dummy_cols = [c for c in df.columns if c.startswith("Preferred foot_")]

Segment
Defense       7369
Midfield      5159
Attack        3210
Goalkeeper    1999
Name: count, dtype: int64 



In [4]:
#Profile and non profile features

common_secondary_extra = ["Age", "Weak foot", "Skill moves", "Weight"] + preferred_foot_dummy_cols + \
                          ["Nation", "League", "Team", "Alternative positions", "play style"]

SEGMENT_CONFIG = {
    "Defense": {
        "primary": ["DEF", "Interceptions", "Def Awareness", "Standing Tackle",
                    "Sliding Tackle", "Jumping", "Stamina", "Aggression", "Height"],
        "shrink": 0.35,
    },
    "Midfield": {
        "primary": ["PAS", "Vision", "Short Passing", "Long Passing", "Curve"],
        "shrink": 0.55,   
    },
    "Attack": {
        "primary": ["PAC", "Acceleration", "Sprint Speed", "SHO", "Finishing",
                     "Shot Power", "Long Shots", "Volleys", "Penalties", "Positioning"],
        "shrink": 0.35,
    },
    "Goalkeeper": {
        "primary": gk_attrs + ["Height"],
        "shrink": None,  
    },
}

all_numeric_pool = list(dict.fromkeys(field_attrs + ["Height"] + common_secondary_extra))


def get_secondary_features(segment: str, primary: list[str]) -> list[str]:
    pool = all_numeric_pool if segment != "Goalkeeper" else []
    return [f for f in pool if f not in primary]


def weak_link_feature(frame: pd.DataFrame, primary: list[str]) -> pd.Series:
    vals = frame[primary].to_numpy(dtype=float)
    return pd.Series(vals.min(axis=1) - vals.mean(axis=1), index=frame.index)

RANDOM_STATE = 42

In [6]:
#Training

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for seg, cfg in SEGMENT_CONFIG.items():
    seg_df = df[df["Segment"] == seg].copy()
    primary_feats = cfg["primary"]
    secondary_feats = get_secondary_features(seg, primary_feats)

    seg_df["weak_link_primary"] = weak_link_feature(seg_df, primary_feats)
    primary_feats_full = primary_feats + ["weak_link_primary"]

    y = seg_df["OVR"].astype(float)
    X_primary = seg_df[primary_feats_full]
    X_secondary = seg_df[secondary_feats] if secondary_feats else None

    idx_train, idx_test = train_test_split(seg_df.index, test_size=0.2, random_state=RANDOM_STATE)

    scaler_p = StandardScaler()
    Xp_train = scaler_p.fit_transform(X_primary.loc[idx_train])
    Xp_test = scaler_p.transform(X_primary.loc[idx_test])
    y_train, y_test = y.loc[idx_train], y.loc[idx_test]

    model_primary = GradientBoostingRegressor(
        n_estimators=150, max_depth=4, learning_rate=0.08, random_state=RANDOM_STATE
    )
    model_primary.fit(Xp_train, y_train)

    train_pred_primary = model_primary.predict(Xp_train)
    test_pred_primary = model_primary.predict(Xp_test)

    model_secondary = None
    scaler_s = None
    shrink = cfg["shrink"]

    if secondary_feats and shrink:
        oof_pred_primary = cross_val_predict(
            GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.08,
                                       random_state=RANDOM_STATE),
            Xp_train, y_train, cv=kf,
        )
        residual_train = y_train.to_numpy() - oof_pred_primary

        scaler_s = StandardScaler()
        Xs_train = scaler_s.fit_transform(X_secondary.loc[idx_train])
        Xs_test = scaler_s.transform(X_secondary.loc[idx_test])

        model_secondary = GradientBoostingRegressor(
            n_estimators=120, max_depth=3, learning_rate=0.06, random_state=RANDOM_STATE
        )
        model_secondary.fit(Xs_train, residual_train)

        train_pred_final = train_pred_primary + shrink * model_secondary.predict(Xs_train)
        test_pred_final = test_pred_primary + shrink * model_secondary.predict(Xs_test)
    else:
        train_pred_final = train_pred_primary
        test_pred_final = test_pred_primary

    train_r2 = r2_score(y_train, train_pred_final)
    test_r2 = r2_score(y_test, test_pred_final)
    test_r2_primary_only = r2_score(y_test, test_pred_primary)

    print(f"--- Segment: {seg} ---")
    print(f"  Primary features   : {primary_feats}")
    print(f"  Secondary count    : {len(secondary_feats)}  | shrink = {shrink}")
    print(f"  Test R2 (only primary)      : {test_r2_primary_only:.4f}")
    print(f"  Test R2 (primary + secondary) : {test_r2:.4f}")

    results[seg] = {"train_r2": train_r2, "test_r2": test_r2}

    joblib.dump(model_primary, f"model_primary_{seg}.pkl")
    joblib.dump(scaler_p, f"scaler_primary_{seg}.pkl")
    joblib.dump(primary_feats_full, f"features_primary_{seg}.pkl")
    if model_secondary is not None:
        joblib.dump(model_secondary, f"model_secondary_{seg}.pkl")
        joblib.dump(scaler_s, f"scaler_secondary_{seg}.pkl")
        joblib.dump(secondary_feats, f"features_secondary_{seg}.pkl")
    joblib.dump(shrink, f"shrink_{seg}.pkl")

print("Summary by segment :")
for seg, r in results.items():
    print(f"  {seg:10s} -> test R2 = {r['test_r2']:.4f}")

--- Segment: Defense ---
  Primary features   : ['DEF', 'Interceptions', 'Def Awareness', 'Standing Tackle', 'Sliding Tackle', 'Jumping', 'Stamina', 'Aggression', 'Height']
  Secondary count    : 37  | shrink = 0.35
  Test R2 (only primary)      : 0.9383
  Test R2 (primary + secondary) : 0.9522
--- Segment: Midfield ---
  Primary features   : ['PAS', 'Vision', 'Short Passing', 'Long Passing', 'Curve']
  Secondary count    : 41  | shrink = 0.55
  Test R2 (only primary)      : 0.8988
  Test R2 (primary + secondary) : 0.9420
--- Segment: Attack ---
  Primary features   : ['PAC', 'Acceleration', 'Sprint Speed', 'SHO', 'Finishing', 'Shot Power', 'Long Shots', 'Volleys', 'Penalties', 'Positioning']
  Secondary count    : 36  | shrink = 0.35
  Test R2 (only primary)      : 0.9392
  Test R2 (primary + secondary) : 0.9496
--- Segment: Goalkeeper ---
  Primary features   : ['GK Diving', 'GK Handling', 'GK Kicking', 'GK Positioning', 'GK Reflexes', 'Height']
  Secondary count    : 0  | shrink = N

In [7]:
#Prediction function
def predict_ovr(segment: str, player_row: pd.DataFrame) -> float:
    """player_row — один DataFrame-ряд с уже очищенными/закодированными признаками."""
    cfg = SEGMENT_CONFIG[segment]
    primary_feats = cfg["primary"]

    row = player_row.copy()
    row["weak_link_primary"] = weak_link_feature(row, primary_feats)
    primary_feats_full = primary_feats + ["weak_link_primary"]

    m_p = joblib.load(f"model_primary_{segment}.pkl")
    s_p = joblib.load(f"scaler_primary_{segment}.pkl")
    pred = m_p.predict(s_p.transform(row[primary_feats_full]))[0]

    shrink = joblib.load(f"shrink_{segment}.pkl")
    if shrink:
        secondary_feats = joblib.load(f"features_secondary_{segment}.pkl")
        m_s = joblib.load(f"model_secondary_{segment}.pkl")
        s_s = joblib.load(f"scaler_secondary_{segment}.pkl")
        pred += shrink * m_s.predict(s_s.transform(row[secondary_feats]))[0]

    return float(np.clip(pred, 1, 99))

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pandas as pd

segments = ['Defense', 'Midfield', 'Attack', 'Goalkeeper']

for seg in segments:
    model = joblib.load(f'model_{seg}.pkl')
    features = joblib.load(f'features_{seg}.pkl')

    feature_importances = model.feature_importances_

    importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': feature_importances
    })

    importance_df = importance_df.sort_values(by='Importance', ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
    plt.title(f'Top 15 Feature Importances for {seg} Segment Model')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    print(f"\nTop features for {seg} segment:")
    for index, row in importance_df.iterrows():
        print(f"- {row['Feature']}: {row['Importance']:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: 'model_Defense.pkl'